# Set up

Retrieve the data we'll use from the git repository: a bunch of markdown files

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


# Q1. How many lesson pages

In [ ]:
print(len(documents))

# Q2. Indexing and Searching

Let's use minsearch to index and search the documents. `index` stores the indexed documents, which have two fields: `filename` and `content`

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

The indexed documents are ready!! Let's run our first search

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

The first result is

In [ ]:
search_results[0]["filename"]

# Q3. RAG Implementation

Create a the `RAGBase` class that will use the `minsearch` index from the previous questions to search for relevant documents and build a prompt. Then it will use the client passed as an argument to query the LLM and return the response.

Initialize the OpenAI client:

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 3.0, 'filename': 0.5}
        filter_dict = {}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text, response.usage.input_tokens

In [ ]:
from openai import OpenAI
import os

from dotenv import load_dotenv

load_dotenv('../.env')

openai_client = OpenAI()


In [ ]:
from pprint import pprint
from IPython.display import display, Markdown

rag = RAGBase(index,openai_client)
# pprint(rag.llm('who are you?'))
answer,input_tokens = rag.rag('How does the agentic loop keep calling the model until it stops?')
display(Markdown(f"We have consummed `{input_tokens}` input tokens"))
display(Markdown(f"**This is the answer that we got:**"))
display(Markdown(answer))

# Q4. Chunking

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
display(Markdown(f"Chunks: `{len(chunks)}`"))

# Q5. RAG with chunking

In [ ]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunked_index.fit(chunks)

rag = RAGBase(chunked_index,openai_client)
# pprint(rag.llm('who are you?'))
answer,chunked_input_tokens = rag.rag('How does the agentic loop keep calling the model until it stops?')
display(Markdown(f"We have consummed `{chunked_input_tokens}` input tokens, **which is about {round(input_tokens/chunked_input_tokens,2)} times less**"))
display(Markdown(f"**This is the answer that we got:**"))
display(Markdown(answer))

# Q6. Turning it into an agent

Define a search function to be used by our agent. By inting the type of the arguments and the return value, and documenting the function, we allow the `aitoykit` tool to infer the json definition of the function that will be passed to the OpenAI client.

We could write this definition instead using this json:

```json
{
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
```

store it in a variable and pass it add tools:

```python
agent_tools.add_tool(search, search_json_definition)
```

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunked_index.search(
        query,
        num_results=5,
        boost_dict={"content": 3.0, "filename": 0.5},
        filter_dict={}
    )

Set up the runner

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

agent_instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)